# Assignment 16: AI-Text Detection Robustness

This notebook is a reproducibility and results-inspection companion to the project scripts. The full experiments are implemented in `src/`; this notebook does not claim to rerun them.

**Research question:** How do surface editing and semantic paraphrasing affect a TF–IDF + LinearSVC detector trained on original HC3 passages?

The main paired evaluation uses 300 held-out question topics, with 300 human passages and 300 AI passages per condition. See `README.md` and `reports/Assignment16_Final_Report.docx` for the project narrative and limitations.


## 1. Environment and project paths

Run this notebook from the repository root, or adjust the paths below. It inspects saved artifacts and does not download data or models.


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists() or not (ROOT / "results").exists():
    raise RuntimeError(
        "Run this notebook from the project root "
        "(the folder containing src/ and results/)."
    )

DATA_PATH = ROOT / "data" / "hc3_original.csv"
RESULTS = ROOT / "results"

print("Project root:", ROOT)
print("Prepared dataset exists:", DATA_PATH.exists())
print("Results directory exists:", RESULTS.exists())


## 2. Inspect the prepared HC3 dataset

The preparation script filters and deduplicates passages and records topic IDs, conditions, labels, and provenance. The following checks are performed only if the prepared CSV is available.


In [ ]:
if not DATA_PATH.exists():
    print("Prepared data not found. Run `python src/prepare_hc3.py` first.")
else:
    df = pd.read_csv(DATA_PATH)
    required = {"passage_id", "topic_id", "condition", "label", "text", "source"}
    missing = required - set(df.columns)
    assert not missing, f"Missing required columns: {sorted(missing)}"
    assert df["passage_id"].is_unique, "passage_id must be unique"
    assert set(df["label"].dropna().astype(int)).issubset({0, 1})

    print("Rows:", len(df))
    print("Distinct topics:", df["topic_id"].nunique())
    print("\nCondition counts:")
    display(df["condition"].value_counts().rename_axis("condition").to_frame("n"))
    print("\nLabel counts:")
    display(df["label"].value_counts().sort_index().rename_axis("label").to_frame("n"))
    print("\nSource by condition:")
    display(pd.crosstab(df["source"], df["condition"]))


## 3. Inspect the main paired evaluation

The main paired results combine human and AI passages in each condition. The separate `robustness_condition_metrics.csv` is a larger, class-specific analysis and must not be interpreted as three balanced evaluations.


In [ ]:
paired_path = RESULTS / "final_condition_metrics.csv"
if paired_path.exists():
    paired = pd.read_csv(paired_path)
    display(paired)
else:
    print("Missing final_condition_metrics.csv; inspect the saved results or run the final evaluation.")


## 4. Inspect the supplementary surface-edit experiment


In [ ]:
robustness_path = RESULTS / "robustness_condition_metrics.csv"
if robustness_path.exists():
    robustness = pd.read_csv(robustness_path)
    display(robustness)
else:
    print("Missing robustness_condition_metrics.csv.")


## 5. Reproduce the experiments

Run these commands manually from the repository root. They can overwrite outputs; preserve validated results before rerunning.

1. `python src/prepare_hc3.py` — prepare `data/hc3_original.csv`.
2. `python src/run_experiment.py` — run the baseline models.
3. `python src/run_robustness.py` — run the larger surface-edit test.
4. `python src/generate_paraphrases.py` — generate semantic paraphrases; requires model download and can take substantial time.
5. `python src/evaluate_final.py` — evaluate the saved paired paraphrases.

The split groups by `topic_id` with random seed 42. The paraphrase evaluation is a 300-topic pilot. It measures the outputs of the tested paraphrasing process and does not establish robustness for every detector or meaning-preserving rewrite.


## 6. Interpretation checklist

- Report accuracy, AI precision, AI recall, AI F1, and false-positive rate.
- Distinguish the 300-topic paired evaluation from the larger class-specific surface-edit evaluation.
- Inspect false positives and false negatives as well as aggregate metrics.
- Report source sample counts; interpret small groups cautiously.
- Some paraphrases changed facts or omitted details, so do not describe every paraphrase as meaning-preserving.
- Detector predictions are not proof of authorship.
